# Step 9: Capacity-Constrained Logistics Optimization Model

This step builds a binary assignment optimization model to assign customer demand zones to fulfillment centers while minimizing transportation cost and satisfying capacity constraints.

In [1]:
# Step 9: Build capacity-constrained logistics optimization model

from pathlib import Path
import sys
import subprocess

import pandas as pd
from IPython.display import display

# Install PuLP if it is not already installed
try:
    import pulp
    print("PuLP is already installed.")
except ImportError:
    print("Installing PuLP...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pulp"])
    import pulp
    print("PuLP installed successfully.")

# Set project directories
project_dir = Path("/Users/mac/Desktop/portfolio2_logistics_optimization")
data_processed_dir = project_dir / "data" / "processed"

# Load optimization input data
distance_cost_matrix = pd.read_csv(data_processed_dir / "distance_cost_matrix.csv")
seller_centers = pd.read_csv(data_processed_dir / "seller_fulfillment_centers_with_capacity.csv")
baseline_metrics = pd.read_csv(data_processed_dir / "baseline_metrics.csv")

print("\n=== Input Table Shapes ===")
print("distance_cost_matrix:", distance_cost_matrix.shape)
print("seller_centers:", seller_centers.shape)
print("baseline_metrics:", baseline_metrics.shape)

# Define customer zones and fulfillment centers
customer_zones = sorted(distance_cost_matrix["customer_zone_id"].unique())
fulfillment_centers = sorted(distance_cost_matrix["seller_center_id"].unique())

print("\n=== Optimization Problem Size ===")
print("Number of customer zones:", len(customer_zones))
print("Number of fulfillment centers:", len(fulfillment_centers))
print("Number of assignment decision variables:", len(customer_zones) * len(fulfillment_centers))

# Create lookup dictionaries
demand_dict = (
    distance_cost_matrix
    .drop_duplicates("customer_zone_id")
    .set_index("customer_zone_id")["demand"]
    .to_dict()
)

capacity_dict = (
    seller_centers
    .set_index("seller_center_id")["warehouse_capacity"]
    .to_dict()
)

cost_dict = (
    distance_cost_matrix
    .set_index(["customer_zone_id", "seller_center_id"])["transportation_cost"]
    .to_dict()
)

distance_dict = (
    distance_cost_matrix
    .set_index(["customer_zone_id", "seller_center_id"])["distance_km"]
    .to_dict()
)

# Create optimization model
model = pulp.LpProblem(
    "capacity_constrained_logistics_assignment",
    pulp.LpMinimize
)

# Decision variable:
# x[c, f] = 1 if customer zone c is assigned to fulfillment center f, otherwise 0
x = pulp.LpVariable.dicts(
    "assign",
    ((c, f) for c in customer_zones for f in fulfillment_centers),
    cat="Binary"
)

# Objective function: minimize total transportation cost
model += pulp.lpSum(
    cost_dict[(c, f)] * x[(c, f)]
    for c in customer_zones
    for f in fulfillment_centers
)

# Constraint 1: each customer zone must be assigned to exactly one fulfillment center
for c in customer_zones:
    model += pulp.lpSum(
        x[(c, f)] for f in fulfillment_centers
    ) == 1, f"assign_once_{c}"

# Constraint 2: assigned demand cannot exceed fulfillment center capacity
for f in fulfillment_centers:
    model += pulp.lpSum(
        demand_dict[c] * x[(c, f)]
        for c in customer_zones
    ) <= capacity_dict[f], f"capacity_{f}"

# Solve optimization model
solver = pulp.PULP_CBC_CMD(msg=False)
model.solve(solver)

solver_status = pulp.LpStatus[model.status]

print("\n=== Solver Result ===")
print("Solver status:", solver_status)
print("Objective value:", pulp.value(model.objective))

# Extract optimized assignment
optimized_rows = []

for c in customer_zones:
    for f in fulfillment_centers:
        if pulp.value(x[(c, f)]) == 1:
            row = distance_cost_matrix[
                (distance_cost_matrix["customer_zone_id"] == c)
                & (distance_cost_matrix["seller_center_id"] == f)
            ].iloc[0].to_dict()
            
            optimized_rows.append(row)

optimized_assignment = pd.DataFrame(optimized_rows)

print("\n=== Optimized Assignment Shape ===")
print("optimized_assignment:", optimized_assignment.shape)

print("\n=== Sample Optimized Assignments ===")
sample_columns = [
    "customer_zone_id",
    "customer_city",
    "customer_state",
    "demand",
    "seller_center_id",
    "seller_city",
    "seller_state",
    "warehouse_capacity",
    "distance_km",
    "transportation_cost"
]

display(optimized_assignment[sample_columns].head(10))

# Calculate fulfillment center load under optimized policy
optimized_center_load = (
    optimized_assignment
    .groupby("seller_center_id", as_index=False)
    .agg(
        assigned_demand=("demand", "sum"),
        assigned_zones=("customer_zone_id", "count")
    )
)

optimized_center_load = optimized_center_load.merge(
    seller_centers[
        [
            "seller_center_id",
            "seller_city",
            "seller_state",
            "warehouse_capacity"
        ]
    ],
    on="seller_center_id",
    how="right"
)

optimized_center_load["assigned_demand"] = optimized_center_load["assigned_demand"].fillna(0).astype(int)
optimized_center_load["assigned_zones"] = optimized_center_load["assigned_zones"].fillna(0).astype(int)

optimized_center_load["utilization_rate"] = (
    optimized_center_load["assigned_demand"] / optimized_center_load["warehouse_capacity"]
)

optimized_center_load["capacity_violation"] = (
    optimized_center_load["assigned_demand"] > optimized_center_load["warehouse_capacity"]
)

optimized_center_load["excess_demand"] = (
    optimized_center_load["assigned_demand"] - optimized_center_load["warehouse_capacity"]
).clip(lower=0)

print("\n=== Optimized Fulfillment Center Load ===")
display(
    optimized_center_load[
        [
            "seller_center_id",
            "seller_city",
            "seller_state",
            "assigned_demand",
            "warehouse_capacity",
            "utilization_rate",
            "capacity_violation",
            "excess_demand",
            "assigned_zones"
        ]
    ].sort_values("assigned_demand", ascending=False)
)

# Calculate optimization metrics
optimized_total_cost = optimized_assignment["transportation_cost"].sum()
optimized_total_demand = optimized_assignment["demand"].sum()
optimized_weighted_avg_distance = (
    optimized_assignment["distance_km"] * optimized_assignment["demand"]
).sum() / optimized_total_demand

optimized_total_capacity = seller_centers["warehouse_capacity"].sum()
optimized_total_excess_demand = optimized_center_load["excess_demand"].sum()
optimized_num_violated_centers = optimized_center_load["capacity_violation"].sum()
optimized_max_utilization = optimized_center_load["utilization_rate"].max()

baseline_total_cost = baseline_metrics.loc[0, "total_transportation_cost"]
baseline_weighted_avg_distance = baseline_metrics.loc[0, "weighted_avg_distance_km"]
baseline_total_excess_demand = baseline_metrics.loc[0, "total_excess_demand"]

cost_change_vs_baseline = (
    optimized_total_cost - baseline_total_cost
) / baseline_total_cost

distance_change_vs_baseline = (
    optimized_weighted_avg_distance - baseline_weighted_avg_distance
) / baseline_weighted_avg_distance

excess_demand_reduction = (
    baseline_total_excess_demand - optimized_total_excess_demand
)

optimization_metrics = pd.DataFrame(
    [
        {
            "policy": "optimized_capacity_constrained_assignment",
            "solver_status": solver_status,
            "total_demand": optimized_total_demand,
            "total_capacity": optimized_total_capacity,
            "total_transportation_cost": optimized_total_cost,
            "weighted_avg_distance_km": optimized_weighted_avg_distance,
            "num_fulfillment_centers": seller_centers.shape[0],
            "num_customer_zones": optimized_assignment.shape[0],
            "num_capacity_violated_centers": optimized_num_violated_centers,
            "total_excess_demand": optimized_total_excess_demand,
            "max_utilization_rate": optimized_max_utilization,
            "cost_change_vs_baseline": cost_change_vs_baseline,
            "distance_change_vs_baseline": distance_change_vs_baseline,
            "excess_demand_reduction_vs_baseline": excess_demand_reduction
        }
    ]
)

print("\n=== Optimization Metrics ===")
display(optimization_metrics)

# Save optimization outputs
assignment_output_path = data_processed_dir / "optimized_assignment.csv"
load_output_path = data_processed_dir / "optimized_center_load.csv"
metrics_output_path = data_processed_dir / "optimization_metrics.csv"

optimized_assignment.to_csv(assignment_output_path, index=False)
optimized_center_load.to_csv(load_output_path, index=False)
optimization_metrics.to_csv(metrics_output_path, index=False)

print("\n=== Step 9 Final Result ===")
print("Saved optimized assignment to:")
print(assignment_output_path)
print("Saved optimized center load to:")
print(load_output_path)
print("Saved optimization metrics to:")
print(metrics_output_path)

Installing PuLP...
   ━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/16.4 MB 47.7 kB/s eta 0:04:27m


Resuming download pulp-3.3.2-py3-none-any.whl (3.7 MB/16.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 16.3/16.4 MB 25.4 kB/s eta 0:00:0600:20m


Resuming download pulp-3.3.2-py3-none-any.whl (16.3 MB/16.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 166.2 kB/s  0:00:000m
PuLP installed successfully.

=== Input Table Shapes ===
distance_cost_matrix: (360, 22)
seller_centers: (12, 11)
baseline_metrics: (1, 10)

=== Optimization Problem Size ===
Number of customer zones: 30
Number of fulfillment centers: 12
Number of assignment decision variables: 360

=== Solver Result ===
Solver status: Infeasible
Objective value: 17701059.42106199

=== Optimized Assignment Shape ===
optimized_assignment: (25, 22)

=== Sample Optimized Assignments ===


,customer_zone_id,customer_city,customer_state,demand,seller_center_id,seller_city,seller_state,warehouse_capacity,distance_km,transportation_cost
0,barueri_SP,barueri,SP,469,FC_13405_piracicaba_SP,piracicaba,SP,3688,120.268723,5.640603e+04
1,belem_PA,belem,PA,474,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4717,2158.334777,1.023051e+06
2,brasilia_DF,brasilia,DF,2157,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4717,575.369162,1.241071e+06
3,campinas_SP,campinas,SP,1622,FC_13405_piracicaba_SP,piracicaba,SP,3688,64.176518,1.040943e+05
4,contagem_MG,contagem,MG,475,FC_14840_guariba_SP,guariba,SP,2708,462.817950,2.198385e+05
5,curitiba_PR,curitiba,PR,1723,FC_14940_ibitinga_SP,ibitinga,SP,17934,413.307191,7.121283e+05
6,florianopolis_SC,florianopolis,SC,647,FC_14940_ibitinga_SP,ibitinga,SP,17934,647.544512,4.189613e+05
7,fortaleza_CE,fortaleza,CE,692,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4717,2229.486297,1.542805e+06
8,goiania_GO,goiania,GO,781,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4717,458.368170,3.579855e+05
9,juiz_de_fora_MG,juiz de fora,MG,468,FC_14940_ibitinga_SP,ibitinga,SP,17934,564.725525,2.642915e+05



=== Optimized Fulfillment Center Load ===


,seller_center_id,seller_city,seller_state,assigned_demand,warehouse_capacity,utilization_rate,capacity_violation,excess_demand,assigned_zones
0,FC_14940_ibitinga_SP,ibitinga,SP,6685,17934,0.372756,False,0,8
2,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4104,4717,0.870045,False,0,4
4,FC_13405_piracicaba_SP,piracicaba,SP,3419,3688,0.927061,False,0,4
3,FC_9015_santo_andre_SP,santo andre,SP,2717,4037,0.673025,False,0,3
11,FC_14840_guariba_SP,guariba,SP,2394,2708,0.884047,False,0,3
1,FC_5849_sao_paulo_SP,sao paulo,SP,830,4732,0.175402,False,0,1
9,FC_13232_campo_limpo_paulista_SP,campo limpo paulista,SP,758,2779,0.272760,False,0,1
6,FC_8577_itaquaquecetuba_SP,itaquaquecetuba,SP,435,3402,0.127866,False,0,1
7,FC_3204_sao_paulo_SP,sao paulo,SP,0,3341,0.000000,False,0,0
5,FC_4782_sao_paulo_SP,sao paulo,SP,0,3506,0.000000,False,0,0



=== Optimization Metrics ===


,policy,solver_status,total_demand,total_capacity,total_transportation_cost,weighted_avg_distance_km,num_fulfillment_centers,num_customer_zones,num_capacity_violated_centers,total_excess_demand,max_utilization_rate,cost_change_vs_baseline,distance_change_vs_baseline,excess_demand_reduction_vs_baseline
0,optimized_capacity_constrained_assignment,Infeasible,21342,56421,1.107255e+07,518.815013,12,25,0,0,0.927061,-0.267464,0.760496,26201



=== Step 9 Final Result ===
Saved optimized assignment to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/optimized_assignment.csv
Saved optimized center load to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/optimized_center_load.csv
Saved optimization metrics to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/optimization_metrics.csv


## Step 9 Result: Infeasible Binary Assignment Model

The initial binary assignment model was infeasible because each customer demand zone had to be assigned to exactly one fulfillment center. Some large customer zones, such as São Paulo and Rio de Janeiro, had demand larger than the capacity of any single fulfillment center.

Therefore, the model was revised in Step 9B as a transportation allocation model, where demand from one customer zone can be split across multiple fulfillment centers.

# Step 9B: Capacity-Constrained Transportation Allocation Model

The binary assignment model was infeasible because some customer zones had demand larger than individual fulfillment center capacities. This step fixes the model by allowing customer-zone demand to be split across fulfillment centers.

In [2]:
# Step 9B: Build capacity-constrained transportation allocation model

from pathlib import Path
import sys
import subprocess

import pandas as pd
from IPython.display import display

# Install PuLP if it is not already installed
try:
    import pulp
    print("PuLP is already installed.")
except ImportError:
    print("Installing PuLP...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pulp"])
    import pulp
    print("PuLP installed successfully.")

# Set project directories
project_dir = Path("/Users/mac/Desktop/portfolio2_logistics_optimization")
data_processed_dir = project_dir / "data" / "processed"

# Load optimization input data
distance_cost_matrix = pd.read_csv(data_processed_dir / "distance_cost_matrix.csv")
seller_centers = pd.read_csv(data_processed_dir / "seller_fulfillment_centers_with_capacity.csv")
baseline_metrics = pd.read_csv(data_processed_dir / "baseline_metrics.csv")

print("\n=== Input Table Shapes ===")
print("distance_cost_matrix:", distance_cost_matrix.shape)
print("seller_centers:", seller_centers.shape)
print("baseline_metrics:", baseline_metrics.shape)

# Define customer zones and fulfillment centers
customer_zones = sorted(distance_cost_matrix["customer_zone_id"].unique())
fulfillment_centers = sorted(distance_cost_matrix["seller_center_id"].unique())

print("\n=== Optimization Problem Size ===")
print("Number of customer zones:", len(customer_zones))
print("Number of fulfillment centers:", len(fulfillment_centers))
print("Number of shipment allocation variables:", len(customer_zones) * len(fulfillment_centers))

# Create lookup dictionaries
demand_dict = (
    distance_cost_matrix
    .drop_duplicates("customer_zone_id")
    .set_index("customer_zone_id")["demand"]
    .to_dict()
)

capacity_dict = (
    seller_centers
    .set_index("seller_center_id")["warehouse_capacity"]
    .to_dict()
)

distance_dict = (
    distance_cost_matrix
    .set_index(["customer_zone_id", "seller_center_id"])["distance_km"]
    .to_dict()
)

# Create optimization model
model = pulp.LpProblem(
    "capacity_constrained_transportation_allocation",
    pulp.LpMinimize
)

# Decision variable:
# y[c, f] = number of orders from customer zone c served by fulfillment center f
y = pulp.LpVariable.dicts(
    "shipment",
    ((c, f) for c in customer_zones for f in fulfillment_centers),
    lowBound=0,
    cat="Continuous"
)

# Objective function: minimize total distance-weighted transportation cost
model += pulp.lpSum(
    distance_dict[(c, f)] * y[(c, f)]
    for c in customer_zones
    for f in fulfillment_centers
)

# Constraint 1: all demand from each customer zone must be served
for c in customer_zones:
    model += pulp.lpSum(
        y[(c, f)] for f in fulfillment_centers
    ) == demand_dict[c], f"demand_fulfillment_{c}"

# Constraint 2: assigned demand cannot exceed fulfillment center capacity
for f in fulfillment_centers:
    model += pulp.lpSum(
        y[(c, f)] for c in customer_zones
    ) <= capacity_dict[f], f"capacity_{f}"

# Solve optimization model
solver = pulp.PULP_CBC_CMD(msg=False)
model.solve(solver)

solver_status = pulp.LpStatus[model.status]

print("\n=== Solver Result ===")
print("Solver status:", solver_status)
print("Objective value:", pulp.value(model.objective))

# Stop if the model is not optimal
if solver_status != "Optimal":
    raise ValueError("Optimization model did not find an optimal solution. Please debug before continuing.")

# Extract optimized allocation
optimized_rows = []

for c in customer_zones:
    for f in fulfillment_centers:
        shipment_qty = pulp.value(y[(c, f)])
        
        if shipment_qty is not None and shipment_qty > 1e-6:
            row = distance_cost_matrix[
                (distance_cost_matrix["customer_zone_id"] == c)
                & (distance_cost_matrix["seller_center_id"] == f)
            ].iloc[0].to_dict()
            
            row["allocated_demand"] = shipment_qty
            row["optimized_transportation_cost"] = row["distance_km"] * shipment_qty
            
            optimized_rows.append(row)

optimized_allocation = pd.DataFrame(optimized_rows)

print("\n=== Optimized Allocation Shape ===")
print("optimized_allocation:", optimized_allocation.shape)

print("\n=== Sample Optimized Allocations ===")
sample_columns = [
    "customer_zone_id",
    "customer_city",
    "customer_state",
    "demand",
    "seller_center_id",
    "seller_city",
    "seller_state",
    "warehouse_capacity",
    "allocated_demand",
    "distance_km",
    "optimized_transportation_cost"
]

display(optimized_allocation[sample_columns].head(15))

# Check whether each customer zone's demand is fully served
customer_demand_check = (
    optimized_allocation
    .groupby("customer_zone_id", as_index=False)
    .agg(
        original_demand=("demand", "first"),
        allocated_demand=("allocated_demand", "sum")
    )
)

customer_demand_check["demand_gap"] = (
    customer_demand_check["original_demand"] - customer_demand_check["allocated_demand"]
)

print("\n=== Customer Demand Fulfillment Check ===")
display(customer_demand_check)

# Calculate fulfillment center load under optimized allocation
optimized_center_load = (
    optimized_allocation
    .groupby("seller_center_id", as_index=False)
    .agg(
        assigned_demand=("allocated_demand", "sum"),
        assigned_zones=("customer_zone_id", "nunique")
    )
)

optimized_center_load = optimized_center_load.merge(
    seller_centers[
        [
            "seller_center_id",
            "seller_city",
            "seller_state",
            "warehouse_capacity"
        ]
    ],
    on="seller_center_id",
    how="right"
)

optimized_center_load["assigned_demand"] = optimized_center_load["assigned_demand"].fillna(0)
optimized_center_load["assigned_zones"] = optimized_center_load["assigned_zones"].fillna(0).astype(int)

optimized_center_load["utilization_rate"] = (
    optimized_center_load["assigned_demand"] / optimized_center_load["warehouse_capacity"]
)

optimized_center_load["capacity_violation"] = (
    optimized_center_load["assigned_demand"] > optimized_center_load["warehouse_capacity"] + 1e-6
)

optimized_center_load["excess_demand"] = (
    optimized_center_load["assigned_demand"] - optimized_center_load["warehouse_capacity"]
).clip(lower=0)

print("\n=== Optimized Fulfillment Center Load ===")
display(
    optimized_center_load[
        [
            "seller_center_id",
            "seller_city",
            "seller_state",
            "assigned_demand",
            "warehouse_capacity",
            "utilization_rate",
            "capacity_violation",
            "excess_demand",
            "assigned_zones"
        ]
    ].sort_values("assigned_demand", ascending=False)
)

# Calculate optimization metrics
optimized_total_cost = optimized_allocation["optimized_transportation_cost"].sum()
optimized_total_demand = optimized_allocation["allocated_demand"].sum()
optimized_weighted_avg_distance = optimized_total_cost / optimized_total_demand

optimized_total_capacity = seller_centers["warehouse_capacity"].sum()
optimized_total_excess_demand = optimized_center_load["excess_demand"].sum()
optimized_num_violated_centers = optimized_center_load["capacity_violation"].sum()
optimized_max_utilization = optimized_center_load["utilization_rate"].max()

baseline_total_cost = baseline_metrics.loc[0, "total_transportation_cost"]
baseline_weighted_avg_distance = baseline_metrics.loc[0, "weighted_avg_distance_km"]
baseline_total_excess_demand = baseline_metrics.loc[0, "total_excess_demand"]

cost_change_vs_baseline = (
    optimized_total_cost - baseline_total_cost
) / baseline_total_cost

distance_change_vs_baseline = (
    optimized_weighted_avg_distance - baseline_weighted_avg_distance
) / baseline_weighted_avg_distance

excess_demand_reduction = (
    baseline_total_excess_demand - optimized_total_excess_demand
)

optimization_metrics = pd.DataFrame(
    [
        {
            "policy": "optimized_capacity_constrained_allocation",
            "solver_status": solver_status,
            "total_demand": optimized_total_demand,
            "total_capacity": optimized_total_capacity,
            "total_transportation_cost": optimized_total_cost,
            "weighted_avg_distance_km": optimized_weighted_avg_distance,
            "num_fulfillment_centers": seller_centers.shape[0],
            "num_customer_zones": len(customer_zones),
            "num_allocation_lanes": optimized_allocation.shape[0],
            "num_capacity_violated_centers": optimized_num_violated_centers,
            "total_excess_demand": optimized_total_excess_demand,
            "max_utilization_rate": optimized_max_utilization,
            "cost_change_vs_baseline": cost_change_vs_baseline,
            "distance_change_vs_baseline": distance_change_vs_baseline,
            "excess_demand_reduction_vs_baseline": excess_demand_reduction
        }
    ]
)

print("\n=== Optimization Metrics ===")
display(optimization_metrics)

# Save optimization outputs
allocation_output_path = data_processed_dir / "optimized_allocation.csv"
load_output_path = data_processed_dir / "optimized_center_load.csv"
metrics_output_path = data_processed_dir / "optimization_metrics.csv"

optimized_allocation.to_csv(allocation_output_path, index=False)
optimized_center_load.to_csv(load_output_path, index=False)
optimization_metrics.to_csv(metrics_output_path, index=False)

print("\n=== Step 9B Final Result ===")
print("Saved optimized allocation to:")
print(allocation_output_path)
print("Saved optimized center load to:")
print(load_output_path)
print("Saved optimization metrics to:")
print(metrics_output_path)

PuLP is already installed.

=== Input Table Shapes ===
distance_cost_matrix: (360, 22)
seller_centers: (12, 11)
baseline_metrics: (1, 10)

=== Optimization Problem Size ===
Number of customer zones: 30
Number of fulfillment centers: 12
Number of shipment allocation variables: 360

=== Solver Result ===
Solver status: Optimal
Objective value: 17701059.422814753

=== Optimized Allocation Shape ===
optimized_allocation: (41, 24)

=== Sample Optimized Allocations ===


,customer_zone_id,customer_city,customer_state,demand,seller_center_id,seller_city,seller_state,warehouse_capacity,allocated_demand,distance_km,optimized_transportation_cost
0,barueri_SP,barueri,SP,469,FC_13405_piracicaba_SP,piracicaba,SP,3688,469.0,120.268723,5.640603e+04
1,belem_PA,belem,PA,474,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4717,474.0,2158.334777,1.023051e+06
2,belo_horizonte_MG,belo horizonte,MG,3077,FC_14840_guariba_SP,guariba,SP,2708,271.0,473.106791,1.282119e+05
3,belo_horizonte_MG,belo horizonte,MG,3077,FC_14940_ibitinga_SP,ibitinga,SP,17934,2806.0,546.489156,1.533449e+06
4,brasilia_DF,brasilia,DF,2157,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4717,2157.0,575.369162,1.241071e+06
5,campinas_SP,campinas,SP,1622,FC_13405_piracicaba_SP,piracicaba,SP,3688,1622.0,64.176518,1.040943e+05
6,contagem_MG,contagem,MG,475,FC_14840_guariba_SP,guariba,SP,2708,475.0,462.817950,2.198385e+05
7,curitiba_PR,curitiba,PR,1723,FC_14940_ibitinga_SP,ibitinga,SP,17934,1723.0,413.307191,7.121283e+05
8,florianopolis_SC,florianopolis,SC,647,FC_14940_ibitinga_SP,ibitinga,SP,17934,647.0,647.544512,4.189613e+05
9,fortaleza_CE,fortaleza,CE,692,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4717,692.0,2229.486297,1.542805e+06



=== Customer Demand Fulfillment Check ===


,customer_zone_id,original_demand,allocated_demand,demand_gap
0,barueri_SP,469,469.0,0.0
1,belem_PA,474,474.0,0.0
2,belo_horizonte_MG,3077,3077.0,0.0
3,brasilia_DF,2157,2157.0,0.0
4,campinas_SP,1622,1622.0,0.0
5,contagem_MG,475,475.0,0.0
6,curitiba_PR,1723,1723.0,0.0
7,florianopolis_SC,647,647.0,0.0
8,fortaleza_CE,692,692.0,0.0
9,goiania_GO,781,781.0,0.0



=== Optimized Fulfillment Center Load ===


,seller_center_id,seller_city,seller_state,assigned_demand,warehouse_capacity,utilization_rate,capacity_violation,excess_demand,assigned_zones
0,FC_14940_ibitinga_SP,ibitinga,SP,12804.0,17934,0.713951,False,0.0,10
1,FC_5849_sao_paulo_SP,sao paulo,SP,4732.0,4732,1.000000,False,0.0,2
2,FC_15025_sao_jose_do_rio_preto_SP,sao jose do rio preto,SP,4717.0,4717,1.000000,False,0.0,5
3,FC_9015_santo_andre_SP,santo andre,SP,4037.0,4037,1.000000,False,0.0,4
4,FC_13405_piracicaba_SP,piracicaba,SP,3688.0,3688,1.000000,False,0.0,5
5,FC_4782_sao_paulo_SP,sao paulo,SP,3506.0,3506,1.000000,False,0.0,1
6,FC_8577_itaquaquecetuba_SP,itaquaquecetuba,SP,3402.0,3402,1.000000,False,0.0,2
7,FC_3204_sao_paulo_SP,sao paulo,SP,3341.0,3341,1.000000,False,0.0,1
8,FC_4160_sao_paulo_SP,sao paulo,SP,2857.0,2857,1.000000,False,0.0,1
9,FC_13232_campo_limpo_paulista_SP,campo limpo paulista,SP,2779.0,2779,1.000000,False,0.0,3



=== Optimization Metrics ===


,policy,solver_status,total_demand,total_capacity,total_transportation_cost,weighted_avg_distance_km,num_fulfillment_centers,num_customer_zones,num_allocation_lanes,num_capacity_violated_centers,total_excess_demand,max_utilization_rate,cost_change_vs_baseline,distance_change_vs_baseline,excess_demand_reduction_vs_baseline
0,optimized_capacity_constrained_allocation,Optimal,51291.0,56421,1.770106e+07,345.110437,12,30,41,0,0.0,1.0,0.171064,0.171064,26201.0



=== Step 9B Final Result ===
Saved optimized allocation to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/optimized_allocation.csv
Saved optimized center load to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/optimized_center_load.csv
Saved optimization metrics to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/optimization_metrics.csv
